In [0]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import pandas as pd

# 1. Load data from your specific Gold View
df = spark.table("instagram.goldlayer.vw_behavioral_user_insights").toPandas()

# Minimal fix: Convert 'biometric_login_used' from 'Yes'/'No' to 1/0
if df['biometric_login_used'].dtype == object:
    df['biometric_login_used'] = df['biometric_login_used'].map({'Yes': 1, 'No': 0})

# 2. Features & Target
# We use the Detailing columns we added
X = df[['age', 'user_engagement_score', 'linked_accounts_count', 'biometric_login_used']]
y = df['stress_category']

# 3. Train with MLflow
with mlflow.start_run(run_name="Stress_Classification"):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    model = RandomForestClassifier(n_estimators=100).fit(X_train, y_train)
    
    mlflow.log_metric("accuracy", model.score(X_test, y_test))
    mlflow.sklearn.log_model(model, "model")
    
    # Save predictions with user_id to join later
    predictions = df[['user_id']].copy()
    predictions['pred_stress_category'] = model.predict(X)
    spark.createDataFrame(predictions).write.mode("overwrite").saveAsTable("instagram.model_output.ml_stress")

print("✅ Stress Model Trained and Temp Table created, bro!")